In [1]:
%pip install -q streamlit scikit-learn pandas joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 47.6 MB/s eta 0:00:00


In [2]:
from google.colab import files
upload = files.upload()

Saving plant_health_pipeline.joblib to plant_health_pipeline.joblib


In [3]:
%%writefile streamlit_app.py

import joblib
import pandas as pd
import streamlit as st

MODEL_PATH = "/content/plant_health_pipeline.joblib"

RESPONSES = {
    "Healthy": "Continue regular care and monitor weekly.",
    "Mild Stress": "Check watering, sunlight, and soil drainage.",
    "Severe Stress": "Provide urgent care and request grounds personnel inspection.",
    "Diseased": "Inspect for disease, remove affected leaves, and isolate if needed."
}

model = joblib.load(MODEL_PATH)

st.set_page_config(
    page_title="Plant Health Classifier",
    page_icon="🌿",
    layout="centered"
)

st.title("🌿 Plant Health Classifier")

st.write(
    "Enter the plant inspection information to classify "
    "the plant's current condition."
)

st.info(
    "This prediction supports campus grounds monitoring. "
    "It is not a final horticultural diagnosis."
)

with st.form("plant_form"):

    plant_name = st.text_input(
        "Plant name (optional)",
        placeholder="Example: Rose bed A"
    )

    leaf_color = st.selectbox(
        "Leaf color",
        ["Light Green", "Pale Green", "Green", "Dark Green", "Yellow", "Brown"]
    )

    spots = st.selectbox(
        "Leaf spots",
        ["None", "Few", "Many"]
    )

    wilting = st.selectbox(
        "Wilting",
        ["None", "Mild", "Severe"]
    )

    soil_moisture = st.number_input(
        "Soil moisture (%)",
        min_value=0.0,
        max_value=100.0,
        value=55.0,
        step=1.0
    )

    growth_rate = st.selectbox(
        "Growth rate",
        ["Slow", "Normal", "Fast"]
    )

    predict_button = st.form_submit_button(
        "Classify Plant Health",
        type="primary",
        use_container_width=True
    )

if predict_button:

    plant_data = pd.DataFrame(
        [[leaf_color, spots, wilting, soil_moisture, growth_rate]],
        columns=[
            "leaf_color",
            "spots",
            "wilting",
            "soil_moisture",
            "growth_rate"
        ]
    )

    try:
        prediction = model.predict(plant_data)[0]

        probabilities = model.predict_proba(plant_data)[0]
        confidence = float(max(probabilities))

        st.divider()
        st.subheader("Classification Result")

        if plant_name.strip():
            st.write(f"**Plant:** {plant_name}")

        if prediction == "Healthy":
            st.success("Prediction: Healthy")

        elif prediction == "Mild Stress":
            st.warning("Prediction: Mild Stress")

        elif prediction == "Severe Stress":
            st.error("Prediction: Severe Stress")

        else:
            st.error("Prediction: Diseased")

        st.write(
            "**Recommended action:** "
            + RESPONSES.get(
                prediction,
                "Request grounds personnel inspection."
            )
        )

        st.metric("Model confidence", f"{confidence:.0%}")

        if confidence < 0.70 or prediction == "Diseased":
            st.warning(
                "Human review recommended before action is taken."
            )

        with st.expander("View submitted information"):
            st.dataframe(
                plant_data,
                use_container_width=True,
                hide_index=True
            )

    except Exception as error:
        st.error("The prediction could not be completed.")
        st.code(str(error))

st.divider()

st.caption(
    "Predictions should be reviewed by campus grounds personnel "
    "before treatment or removal decisions."
)

Writing streamlit_app.py


In [4]:
!pkill -f streamlit || true

!streamlit run /content/streamlit_app.py \
  --server.port 8502 \
  --server.address 0.0.0.0 \
  --server.headless true \
  --server.enableCORS false \
  --server.enableXsrfProtection false \
  > /content/streamlit_log_8502.txt 2>&1 &

^C


In [5]:
import time

time.sleep(12)

!curl -I http://localhost:8502
!cat /content/streamlit_log_8502.txt

HTTP/1.1 200 OK
date: Mon, 12 Oct 2026 13:30:31 GMT
server: uvicorn
content-type: text/html; charset=utf-8
accept-ranges: bytes
content-length: 6463
last-modified: Mon, 12 Oct 2026 13:29:58 GMT
etag: "783e7e0543789293d4755e7b3bdc4ad0"
cache-control: no-cache



2026-10-12 13:30:21.366 Uvicorn server started on 0.0.0.0:8502

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8502
  Network URL: http://172.28.0.12:8502
  External URL: http://35.237.101.83:8502



In [6]:
from google.colab.output import eval_js

app_url = eval_js("google.colab.kernel.proxyPort(8502)")
print("Open this link:", app_url)

Open this link: https://8502-m-s-kkb-use1b1-1hu0tbymvky67-b.us-east1-1.prod.colab.dev
